# NB01 · Explorer et préparer des données

**R5-07 · Introduction à l'intelligence artificielle · Séance 1**

*© Romain Dulout, UPPA. Supports de cours originaux : toute reproduction ou réutilisation, même partielle, nécessite l'accord de l'auteur.*

Un modèle d'IA n'est jamais meilleur que les données qu'on lui donne. Avant d'entraîner quoi que ce soit, un ou une data scientist passe la majorité de son temps à **comprendre** et **préparer** ses données. C'est l'objet de ce notebook, sur un jeu de données réel : le prix de ~54 000 diamants.

### Objectifs
À la fin de ce notebook, vous serez capable de :
1. faire le **diagnostic** d'un jeu de données (taille, types, valeurs manquantes, doublons, erreurs de saisie) ;
2. l'**explorer visuellement** pour repérer des relations et des pièges ;
3. le **nettoyer** en justifiant chaque choix ;
4. **encoder** les variables textuelles en nombres ;
5. **découper** les données en entraînement / test et les **normaliser** sans fuite de données.

### Organisation
- Le **socle** (exercices 0 à 8) est faisable par tout le monde dans la séance. C'est lui qui compte.
- Les **bonus**, notés de ⭐ à ⭐⭐⭐ selon leur difficulté, sont pour celles et ceux qui vont plus vite. Ils ne sont jamais évalués.
- Après chaque exercice, une cellule `grader.check(...)` vérifie votre réponse : elle indique si c'est juste, ou donne un indice pour corriger. On peut la relancer autant de fois que nécessaire.
- Les cellules **Prédire avant d'exécuter** vous demandent de deviner un résultat *avant* de le calculer. L'intérêt est de s'engager sur une réponse avant de voir le résultat.
- Les indices donnent souvent seulement le **nom d'une méthode**. Pour savoir ce qu'elle fait et comment l'utiliser, placez le curseur sur son nom et appuyez sur **Maj + Tab**, ou exécutez `help(...)` (par exemple `help(pd.DataFrame.dropna)`). La documentation de pandas en ligne fonctionne aussi.

Dans le code, `...` signifie **à compléter**.

## Exercice 0 · Démarrer

Exécutez la cellule ci-dessous (Maj + Entrée) après avoir écrit **votre prénom** entre les guillemets. Il sert à tirer **votre propre échantillon** de 5 000 diamants : gardez toujours le même prénom pendant tout le module.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import otter

import course_tools as ct

grader = otter.Notebook(tests_dir="tests")
sns.set_theme()

ct.check_install()
ct.start("...")   # ← écrivez votre prénom ici, par exemple ct.start("Camille")

### Les données dans un tableau : pandas

Pour manipuler des données en Python, on utilise presque toujours la bibliothèque **pandas** (importée plus haut sous le nom `pd`). Elle range les données dans un **DataFrame** : un tableau avec des lignes et des colonnes nommées, comme une feuille de tableur, mais que l'on manipule avec du code.

C'est l'outil de base du machine learning, pour trois raisons :
- on charge presque n'importe quelle source en une ligne : un fichier CSV (`pd.read_csv`), un fichier Excel (`pd.read_excel`), une base de données SQL (`pd.read_sql`)… ;
- on sélectionne, filtre, trie, corrige ou résume ensuite les données en quelques commandes, sans écrire de boucle ;
- les bibliothèques de graphiques (seaborn) et de machine learning (scikit-learn) acceptent directement un DataFrame.

Par convention, on appelle souvent son DataFrame `df`. La cellule suivante charge votre échantillon dans `df` (la fonction du cours appelle `pd.read_csv` pour vous), puis `df.head()` affiche ses 5 premières lignes.

In [ ]:
df = ct.load_dirty_diamonds()
df.head()

Chaque **ligne** est un diamant (une *observation*). Chaque **colonne** est une caractéristique (une *variable*) :

| Colonne | Signification |
|---|---|
| `carat` | poids du diamant (1 carat = 0,2 g) |
| `cut` | qualité de la taille : Fair < Good < Very Good < Premium < Ideal |
| `color` | couleur : de J (la moins bonne) à D (la meilleure) |
| `clarity` | pureté : I1 (la moins bonne) < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF (la meilleure) |
| `depth` | profondeur totale en % |
| `table` | largeur du sommet en % |
| `price` | **prix en dollars : c'est ce qu'on voudra prédire plus tard (la *cible*)** |
| `x`, `y`, `z` | longueur, largeur, hauteur en mm |

## Exercice 1 · Premier contact

Trois commandes à connaître par cœur pour découvrir un jeu de données :
- `df.shape` : (nombre de lignes, nombre de colonnes) ;
- `df.info()` : le type de chaque colonne et le nombre de valeurs non vides ;
- `df.describe()` : les statistiques des colonnes numériques (moyenne, min, max…).

In [ ]:
df.info()

In [ ]:
df.describe()

**À vous.** Complétez les trois variables :
- `n_rows` : le nombre de lignes du tableau, c'est-à-dire le nombre de diamants ;
- `n_columns` : le nombre de colonnes, c'est-à-dire le nombre de variables ;
- `n_numeric` : le nombre de colonnes **numériques**, celles qui contiennent des nombres et non du texte.

Indices :
- `df.shape` donne deux nombres à la fois : comment récupérer l'un ou l'autre ?
- une colonne est *numérique* si son type, affiché par `df.info()`, est `float64` ou `int64`. La méthode `select_dtypes` filtre les colonnes d'un tableau selon leur type : à vous de trouver comment ne garder que les nombres, puis de compter les colonnes restantes.

In [ ]:
n_rows = ...
n_columns = ...
n_numeric = ...

In [ ]:
grader.check("q1")

## Exercice 2 · Diagnostic : qu'est-ce qui cloche ?

Un jeu de données réel contient presque toujours des défauts. Les trois plus fréquents :
- des **valeurs manquantes** (cases vides, `NaN`) : la méthode `isnull` les repère, colonne par colonne (la cellule plus bas vous montre le résultat) ;
- des **doublons** (la même ligne deux fois) : la méthode `duplicated` les repère ;
- des **erreurs de saisie** dans les colonnes texte : on les repère en regardant les valeurs distinctes d'une colonne.

Pour les valeurs distinctes, deux commandes s'appliquent à **une colonne**, que l'on sélectionne avec `df["nom_de_la_colonne"]`. Elles se ressemblent mais ne répondent pas à la même question :
- `.value_counts()` **liste** chaque valeur distincte avec son nombre d'apparitions : c'est elle qui permet de repérer une faute ;
- `.nunique()` donne seulement le **nombre** de valeurs distinctes : un seul nombre, sans le détail.

> **Prédire avant d'exécuter** : d'après le tableau des colonnes ci-dessus, combien de valeurs différentes devrait contenir la colonne `cut` ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
df["cut"].value_counts()

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

**À vous.** Complétez :
- `n_missing` : le nombre **total** de valeurs manquantes dans tout le tableau (et non colonne par colonne) ;
- `n_duplicates` : le nombre de lignes en double ;
- `n_cut_categories` : le nombre de valeurs distinctes (*modalités*) de la colonne `cut`, telles qu'elles apparaissent.

In [ ]:
df.isnull().sum()

In [ ]:
n_missing = ...
n_duplicates = ...
n_cut_categories = ...

In [ ]:
grader.check("q2")

## Exercice 3 · Explorer avec des graphiques

Les chiffres ne disent pas tout : un graphique révèle en une seconde une relation ou une anomalie. Quatre graphiques de base avec `seaborn` :
- `sns.histplot` : la **distribution** d'une variable ;
- `sns.scatterplot` : la **relation** entre deux variables numériques ;
- `sns.boxplot` : comparer une variable numérique **entre catégories** ;
- `sns.heatmap` : la **carte des corrélations**, souvent le premier graphique que l'on trace.

**La heatmap** est un tableau coloré qui croise toutes les variables numériques. Chaque case contient la **corrélation** entre deux variables, un nombre entre -1 et 1 :
- proche de **1** : les deux variables augmentent ensemble ;
- proche de **-1** : quand l'une augmente, l'autre diminue ;
- proche de **0** : pas de lien en ligne droite entre les deux.

En lisant la ligne `price`, on voit d'un coup d'œil quelles variables sont les plus liées au prix, donc lesquelles aideront un modèle à le prédire.

In [ ]:
sns.histplot(data=df, x="price", bins=50)
plt.title("Distribution des prix")
plt.show()

La plupart des diamants valent moins de 5 000 $, quelques-uns beaucoup plus : la distribution est **asymétrique**. On s'en souviendra pour la régression.

> **Prédire avant d'exécuter** : parmi les variables numériques (`carat`, `depth`, `table`, `x`, `y`, `z`), laquelle est la plus liée au prix ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
sns.scatterplot(data=df, x="carat", y="price", alpha=0.3)
plt.title("Prix en fonction du poids")
plt.show()

sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Corrélations entre variables numériques")
plt.show()

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

**Remarque** : la corrélation entre `carat` et `price` est très faible dans la heatmap, alors qu'on s'attendrait à ce que le poids compte énormément. Regardez l'axe horizontal du nuage de points : des diamants de **plusieurs centaines de carats** ? Le plus gros diamant taillé du monde en fait 545… Quelques valeurs aberrantes suffisent à écraser une corrélation. On réglera ça à l'exercice 4.

> **Prédire avant d'exécuter** : la colonne `cut` note la qualité de la taille du diamant, de `Fair` (la moins bonne) à `Ideal` (la meilleure). À votre avis, parmi ces cinq catégories, les diamants `Ideal` sont-ils **en moyenne** les plus chers ou les moins chers ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
valid_cuts = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
df_valid = df[df["cut"].isin(valid_cuts)]

sns.boxplot(data=df_valid, x="cut", y="price", order=valid_cuts)
plt.title("Prix selon la qualité de taille")
plt.show()

df_valid.groupby("cut")["price"].mean().sort_values()

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

**À vous.** Complétez :
- `most_correlated_variable` : le nom (entre guillemets) de la variable numérique la plus corrélée au prix **d'après la heatmap**, hors `price` ;
- `cheapest_cut` : la taille (parmi les 5 valides) dont le prix moyen est le plus bas.

Puis répondez à la question dans la cellule de texte.

In [ ]:
most_correlated_variable = "..."
cheapest_cut = "..."

In [ ]:
grader.check("q3")

> **Pourquoi les diamants `Ideal` sont-ils en moyenne moins chers ?** Indice : comparez le poids moyen (`carat`) de chaque qualité de taille, en vous inspirant du `groupby` de la cellule du boxplot.
>
> **Ma réponse :** *(double-cliquez ici)*

In [ ]:
# Faites ici le calcul qui vous aide à répondre

## Exercice 4 · Nettoyer

Nettoyer, c'est **décider** quoi faire de chaque défaut. Il n'y a pas toujours une seule bonne réponse, mais il faut toujours une **justification**. Pour le socle, on suit tous la même recette, en 5 étapes, dans un nouveau tableau `df_clean`.

| Défaut | Décision | Pourquoi |
|---|---|---|
| a. Doublons | les supprimer | une même ligne compterait deux fois |
| b. Fautes de saisie dans `cut` | les corriger | « Premuim » et « Premium » sont la même chose |
| c. Prix manquant | supprimer la ligne | sans prix, le diamant ne peut pas servir à apprendre le prix |
| d. Poids en milligrammes, dimensions à 0 ou aberrantes | corriger le poids, supprimer les dimensions impossibles | un diamant de 0 mm ou de 58 mm de large n'existe pas |
| e. Autres valeurs manquantes | les remplacer (*imputer*) | on garde la ligne : la médiane pour les nombres, la valeur la plus fréquente pour `color` |

### a. Supprimer les doublons
Méthode utile : `drop_duplicates`.

In [ ]:
df_clean = df.copy()
df_clean = ...
print(len(df), "→", len(df_clean), "lignes")

### b. Corriger les fautes de saisie dans `cut`
Objectif : que `cut` ne contienne plus que les **5 catégories valides** (Fair, Good, Very Good, Premium, Ideal), au lieu des 8 valeurs distinctes trouvées à l'exercice 2.

La première ligne enlève les espaces en trop au début et à la fin des valeurs (`str.strip`). Il reste deux fautes de frappe : retrouvez-les dans le `value_counts()` de l'exercice 2, puis corrigez-les avec la méthode `replace`.

In [ ]:
df_clean["cut"] = df_clean["cut"].str.strip()
df_clean["cut"] = df_clean["cut"].replace({...})   # ← un dictionnaire {"faute": "correction", ...}
df_clean["cut"].value_counts()

### c. Supprimer les lignes sans prix
Méthode utile : `dropna`. Attention, sans précision elle supprime les lignes où **n'importe quelle** colonne est vide : cherchez le paramètre qui permet de ne regarder que la colonne `price`.

In [ ]:
df_clean = ...
df_clean["price"].isnull().sum()

### d. Poids en milligrammes et dimensions impossibles
Certains poids ont été saisis en **milligrammes** au lieu de carats (1 carat = 200 mg). Aucun diamant de l'échantillon ne dépasse 6 carats : toute valeur **supérieure à 10** est donc un poids en mg, à diviser par 200.

Ensuite, on ne garde que les diamants dont les trois dimensions `x`, `y`, `z` sont **strictement positives** et dont `y` et `z` sont **inférieurs à 30 mm**.

**Modifier seulement certaines lignes avec `.loc`.** On procède en deux temps :
1. `in_mg = df_clean["carat"] > 10` crée un **masque** : une colonne de `True` / `False`, qui vaut `True` pour les diamants dont le poids dépasse 10 ;
2. `df.loc[lignes, colonne]` désigne une partie du tableau : les lignes d'abord, la colonne ensuite. Avec un masque à la place des lignes, on ne désigne que les lignes où le masque vaut `True`. La même écriture sert à lire ces cases et à les remplacer avec `=` : seules ces cases changent, le reste du tableau ne bouge pas.

Pour les dimensions, on combine plusieurs conditions avec `&` (« et »), chacune entre parenthèses.

In [ ]:
in_mg = df_clean["carat"] > 10
df_clean.loc[in_mg, "carat"] = ...

dimensions_ok = (df_clean["x"] > 0) & ...   # ← complétez les autres conditions avec &
df_clean = df_clean[dimensions_ok]
df_clean[["carat", "x", "y", "z"]].describe()

### e. Imputer les valeurs manquantes restantes
- pour `carat`, `depth` et `table` : remplacer par la **médiane** de la colonne ;
- pour `color` : remplacer par la valeur **la plus fréquente** (le *mode*).

Méthodes utiles, qui s'appliquent à une colonne : `median`, `mode` et `fillna`. Attention, `mode` ne renvoie pas une valeur mais une **liste** de valeurs (il peut y avoir des ex aequo).

In [ ]:
for column in ["carat", "depth", "table"]:
    df_clean[column] = df_clean[column].fillna(...)

df_clean["color"] = ...

df_clean.isnull().sum()

In [ ]:
grader.check("q4")

> **Pourquoi la médiane plutôt que la moyenne ?** Pensez aux valeurs extrêmes vues dans l'histogramme.
>
> **Ma réponse :** *(double-cliquez ici)*

> **Prédire avant d'exécuter** : maintenant que les données sont propres, que vaut la corrélation entre `carat` et `price` ? Plus forte ou plus faible qu'avant le nettoyage ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
df_clean.select_dtypes("number").corr()["price"].round(2)

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

## Exercice 5 · Encoder les variables textuelles

Un modèle ne sait faire que des calculs : il ne comprend pas « Premium » ou « VS1 ». Il faut transformer le texte en nombres. Deux grandes méthodes :

- **Encodage ordinal** : quand les catégories ont un **ordre** naturel, on les numérote dans cet ordre (Fair = 0, Good = 1, … Ideal = 4).
- **Encodage one-hot** : quand il n'y a **pas d'ordre** (par exemple des noms de villes), on crée une colonne 0/1 par catégorie.

Ici, `cut`, `color` et `clarity` ont toutes un ordre : on utilise l'encodage ordinal.

> **Prédire avant d'exécuter** : si on utilisait plutôt l'encodage **one-hot** pour les trois colonnes (`cut` : 5 catégories, `color` : 7, `clarity` : 8), combien de colonnes aurait le tableau au total ? (il y a 7 colonnes numériques en plus)
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
n_columns_onehot = pd.get_dummies(df_clean, columns=["cut", "color", "clarity"]).shape[1]
n_columns_onehot

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

**À vous.** Complétez les listes dans l'ordre, **de la moins bonne à la meilleure** qualité (regardez le tableau de l'exercice 0), puis exécutez.

In [ ]:
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = [...]      # ← de J à D
clarity_order = [...]    # ← de I1 à IF

df_enc = df_clean.copy()
for column, order in [("cut", cut_order), ("color", color_order), ("clarity", clarity_order)]:
    mapping = {category: number for number, category in enumerate(order)}
    df_enc[column] = df_enc[column].map(mapping)

df_enc.head()

In [ ]:
grader.check("q5")

## Exercice 6 · Découper puis normaliser

Pour savoir si un modèle fonctionne vraiment, on le teste sur des données **qu'il n'a jamais vues**. On coupe donc le jeu en deux :
- **entraînement** (*train*, 80 %) : pour apprendre ;
- **test** (20 %) : pour évaluer, comme un examen final.

On sépare aussi les **variables d'entrée** `X` (tout sauf le prix) de la **cible** `y` (le prix).

Ensuite, on **normalise** : certaines variables vont de 0 à 5 (`carat`), d'autres de 40 à 80 (`depth`). Beaucoup de modèles sont perturbés par ces échelles différentes. `StandardScaler` ramène chaque colonne à une moyenne de 0 et un écart-type de 1.

**Règle d'or** : le scaler apprend les moyennes et écarts-types **sur l'entraînement uniquement** (`fit`), puis on applique la même transformation au test (`transform`). Le test doit rester « inconnu ».

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_enc.drop(columns=...)   # ← toutes les colonnes sauf la cible
y = df_enc[...]                # ← la cible

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=..., random_state=ct.seed())
print(X_train.shape, X_test.shape)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(...)   # ← fit + transform sur l'entraînement
X_test_sc = scaler.transform(...)        # ← seulement transform sur le test

> **Prédire avant d'exécuter** : la moyenne de chaque colonne de `X_train_sc` sera-t-elle exactement 0 ? Et celle de `X_test_sc` ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
print("Moyennes train :", X_train_sc.mean(axis=0).round(3))
print("Moyennes test  :", X_test_sc.mean(axis=0).round(3))

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

In [ ]:
grader.check("q6")

## Exercice 7 · La fuite de données

Une **fuite de données** (*data leakage*) se produit quand une information du jeu de test se glisse, même indirectement, dans la préparation des données ou dans l'entraînement. Le test n'est alors plus vraiment « inconnu », et le score obtenu dessus est trop optimiste. C'est une erreur d'autant plus traître que le code tourne sans aucun message d'erreur.

Voici le code qu'un collègue (ou une IA…) a écrit. Il fonctionne, et pourtant il contient une fuite. À vous de la repérer.

In [ ]:
# Code du collègue
colleague_scaler = StandardScaler()
X_all_scaled = colleague_scaler.fit_transform(X)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_all_scaled, y, test_size=0.2, random_state=0)

Quel est le problème ?

- **A.** Aucun : le résultat est identique à celui de l'exercice 6, aux arrondis près.
- **B.** Le scaler a été ajusté sur **toutes** les données, test compris, avant le découpage.
- **C.** Il fallait aussi normaliser `y`, sinon le modèle mélange des échelles différentes.
- **D.** Avec `random_state=0`, le découpage est toujours le même : à force, le modèle finit par connaître le test.

In [ ]:
answer_leak = "..."   # ← "A", "B", "C" ou "D"

In [ ]:
grader.check("q7")

### Voir la fuite sur un petit exemple

Sur 5 000 diamants, l'effet de la fuite est trop petit pour se voir. Prenons un exemple volontairement exagéré : 8 petits diamants pour l'entraînement, et 2 très gros diamants dans le test. On normalise leur poids de deux façons :
- **bonne pratique** : le scaler apprend la moyenne et l'écart-type sur le train seul (`fit` sur le train), puis transforme le train et le test ;
- **avec fuite** : comme le collègue, le scaler apprend sur le train et le test réunis.

Cette cellule est complète, exécutez-la.

In [ ]:
# 8 petits diamants pour l'entraînement, 2 très gros pour le test (poids en carats)
carat_train = np.array([[0.3], [0.4], [0.5], [0.5], [0.7], [0.8], [0.9], [1.0]])
carat_test = np.array([[2.5], [3.0]])

# Bonne pratique : le scaler apprend uniquement sur le train
good_scaler = StandardScaler().fit(carat_train)

# Avec fuite : le scaler apprend sur le train ET le test
leaky_scaler = StandardScaler().fit(np.vstack([carat_train, carat_test]))

fig, axes = plt.subplots(2, 1, figsize=(9, 3.5), sharex=True)
for ax, demo_scaler, title in [(axes[0], good_scaler, "Bonne pratique : fit sur le train seul"),
                               (axes[1], leaky_scaler, "Avec fuite : fit sur le train et le test")]:
    ax.scatter(demo_scaler.transform(carat_train), [0] * 8, s=80, label="train")
    ax.scatter(demo_scaler.transform(carat_test), [0] * 2, s=120, marker="X", label="test")
    ax.axvline(0, color="grey", linestyle="--")
    ax.set_title(title)
    ax.set_yticks([])
axes[0].legend(loc="upper center", ncol=2)
axes[1].set_xlabel("poids normalisé")
plt.tight_layout()
plt.show()

> **À observer.**
> - **En haut**, le train est centré sur 0 et les deux diamants du test apparaissent très loin à droite : ils sont hors norme par rapport à tout ce que le modèle a vu. C'est exactement la situation qu'il rencontrera en vrai.
> - **En bas**, les deux gros diamants du test ont tiré la moyenne vers le haut et élargi l'écart-type : tout le train se retrouve décalé à gauche de 0, et les diamants du test paraissent presque ordinaires. La préparation du train a été calculée en « sachant » que de gros diamants allaient arriver : c'est cela, la fuite.

> **En une phrase : pourquoi une fuite de données est-elle dangereuse ?** Pensez à ce qui se passe quand le modèle est utilisé « pour de vrai », sur de nouveaux diamants.
>
> **Ma réponse :** *(double-cliquez ici)*

## Exercice 8 · Sauvegarder pour la prochaine séance

Votre tableau propre et encodé `df_enc` resservira à la séance de régression. On le sauvegarde dans votre dossier personnel.

In [ ]:
path = ct.personal_folder() / "diamonds_prepare.csv"
df_enc.to_csv(path, index=False)
print("Sauvegardé :", path)

In [ ]:
grader.check("q8")

---
# Fin du socle

Vous savez maintenant faire ce que fait un ou une data scientist au début de chaque projet. Les bonus qui suivent sont plus ouverts et moins guidés.

## ⭐ Bonus 1 · Enquête libre

Pas de consigne pas à pas ici : à vous de choisir les bons graphiques et calculs. Utilisez `df_clean`.

1. Le **prix au carat** (`price / carat`) dépend-il de la pureté (`clarity`) ? Montrez-le avec un graphique.
2. Trouvez **une relation surprenante** dans les données, montrez-la avec un graphique et expliquez-la en deux phrases.
3. Les colonnes `x`, `y` et `z` apportent-elles une information différente de `carat` ? Argumentez avec un calcul.

In [ ]:
# Votre enquête ici (ajoutez autant de cellules que vous voulez)

> **Mes conclusions :** *(double-cliquez ici)*

## ⭐⭐ Bonus 2 · Tout refaire en une seule chaîne : le `Pipeline`

En entreprise, on ne nettoie pas « à la main » cellule par cellule : on construit une **chaîne de traitement** réutilisable, qu'on peut appliquer à de nouvelles données. Dans scikit-learn, c'est le rôle de `Pipeline` et `ColumnTransformer`.

**Défi.** Construisez un `ColumnTransformer` nommé `preprocessing` qui, en une seule commande :
- impute par la médiane puis normalise les colonnes numériques (`carat`, `depth`, `table`, `x`, `y`, `z`) ;
- impute par le mode puis encode de façon ordinale `cut`, `color`, `clarity`.

Appliquez-le à `X_raw_train` (ci-dessous) avec `fit_transform`, puis à `X_raw_test` avec `transform`, et rangez les résultats dans `Xp_train` et `Xp_test`.

Documentation : `sklearn.compose.ColumnTransformer`, `sklearn.pipeline.make_pipeline`, `sklearn.impute.SimpleImputer`, `sklearn.preprocessing.OrdinalEncoder` (paramètre `categories`).

In [ ]:
# Données de départ : nettoyées (étapes a à d) mais pas imputées ni encodées
raw = df.drop_duplicates()
raw["cut"] = raw["cut"].str.strip().replace({"Premuim": "Premium", "ideal": "Ideal"})
raw = raw.dropna(subset=["price"])
raw.loc[raw["carat"] > 10, "carat"] = raw.loc[raw["carat"] > 10, "carat"] / 200
raw = raw[(raw[["x", "y", "z"]] > 0).all(axis=1) & (raw["y"] < 30) & (raw["z"] < 30)]

X_raw = raw.drop(columns="price")
y_raw = raw["price"]
X_raw_train, X_raw_test, y_raw_train, y_raw_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=ct.seed())

# À vous : construisez `preprocessing`, puis calculez Xp_train et Xp_test

In [ ]:
grader.check("bonus2")

## ⭐⭐⭐ Bonus 3 · Le fichier saboté

Le fichier `diamonds_sabote.csv` contient 3 000 diamants qui semblent propres : aucune valeur manquante, aucun doublon… et pourtant **5 problèmes** y ont été cachés. À vous de les trouver, sans aide.

In [ ]:
sabotaged = pd.read_csv(ct._root() / "donnees" / "diamonds_sabote.csv")
sabotaged.head()

In [ ]:
# Votre enquête ici

Parmi ces 10 problèmes possibles, lesquels sont **réellement** présents ? (il y en a exactement 5)

1. Des prix négatifs
2. Des dates dans le futur
3. Une profondeur (`depth`) incohérente avec les dimensions `x`, `y`, `z`
4. Des valeurs manquantes codées par -999
5. Une modalité de `clarity` qui n'existe pas
6. Des largeurs `table` supérieures à 100 %
7. Des prix saisis en milliers de dollars
8. Des diamants de poids nul
9. Une colonne en double sous un autre nom
10. Des couleurs écrites en minuscules

Indice pour le n° 3 : pour un diamant rond, `depth` ≈ `2 * z / (x + y) * 100`.

In [ ]:
problems_found = [...]   # ← par exemple [1, 4, 6, 8, 10]

In [ ]:
grader.check("bonus3")